# Milvus Track Catalog Indexing

This notebook builds one Milvus collection that stores:

- all track metadata fields from `TalkPlayData-Challenge-Track-Metadata`
- all six embedding fields from `TalkPlayData-Challenge-Track-Embeddings`
- one benchmark-compatible BM25 text/sparse pair and one sparse BM25 pair per experimental metadata field

Start Milvus first:

```bash
docker compose -f docker-compose.milvus.yml up -d
```


In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "mcrs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from mcrs.milvus.indexing import (
    build_track_collection_plan,
    connect_milvus,
    insert_track_documents,
    iter_track_documents,
    load_track_embedding_rows,
    load_track_metadata_rows,
    recreate_track_collection,
    wait_for_collection_indexes,
)


In [ ]:
MILVUS_URI = "http://localhost:19530"
DB_NAME = "default"
COLLECTION_NAME = "music_track_catalog"
DROP_EXISTING = True
BATCH_SIZE = 64


## Load datasets and infer the collection schema

In [ ]:
metadata_rows = load_track_metadata_rows()
embedding_rows = load_track_embedding_rows()
plan = build_track_collection_plan(metadata_rows, embedding_rows)

field_table = pd.DataFrame([
    {
        "name": field.name,
        "datatype": field.datatype_name,
        "nullable": field.nullable,
        "is_primary": field.is_primary,
        "max_length": field.max_length,
        "element_type": field.element_type_name,
        "max_capacity": field.max_capacity,
        "dim": field.dim,
    }
    for field in plan.fields
])

print(f"Metadata rows: {plan.metadata_row_count:,}")
print(f"Metadata-only rows: {plan.metadata_only_row_count:,}")
print(f"BM25 text fields: {', '.join(plan.bm25_field_names)}")
display(field_table)
display(pd.DataFrame([
    {"vector_field": name, "present_rows": count}
    for name, count in plan.embedding_presence_counts.items()
]))


## Create the collection and insert the catalog

In [ ]:
client = connect_milvus(uri=MILVUS_URI, db_name=DB_NAME)
recreate_track_collection(
    client=client,
    collection_name=COLLECTION_NAME,
    collection_plan=plan,
    drop_existing=DROP_EXISTING,
)
inserted_rows = insert_track_documents(
    client=client,
    collection_name=COLLECTION_NAME,
    documents=iter_track_documents(metadata_rows, embedding_rows, vector_dims=plan.vector_dims),
    batch_size=BATCH_SIZE,
)
client.flush(collection_name=COLLECTION_NAME)
wait_for_collection_indexes(client=client, collection_name=COLLECTION_NAME)
client.load_collection(collection_name=COLLECTION_NAME)
print(f"Inserted rows: {inserted_rows:,}")


## Verify the created collection

In [ ]:
description = client.describe_collection(collection_name=COLLECTION_NAME)
description


In [ ]:
sample_track_id = metadata_rows[0]["track_id"]
client.query(
    collection_name=COLLECTION_NAME,
    filter=f'track_id == "{sample_track_id}"',
    limit=1,
    output_fields=["track_name", "artist_name", "tag_list", "bm25_compat_text", "tag_list_text", "metadata_qwen3_embedding_0_6b"],
)
